# Chapter 5: Who Runs the Show

This notebook is the reader-facing path for Chapter 5, *Who Runs the Show: Orchestration Patterns and Model Routing*. It is not a replacement for the prose. It is the practical map that keeps the chapter's orchestration decisions and code examples in one place.

## Start Here: how to use this notebook

Run the notebook from top to bottom once. Most cells run offline with deterministic teaching doubles. The examples preserve the control-flow shape from the OpenAI Agents SDK snippets in the chapter, but they do not require an API key.

### What you will build

1. A code-led invoice pipeline where Python owns the consequence of a model extraction.
2. A loop controller with explicit final-output, handoff, exception, and max-turn endings.
3. A compound-reliability table and a checkpointed workflow that resumes after a crash.
4. A front-desk request router with an `other` category, confidence threshold, and human fallback.
5. A router evaluation with confusion-matrix, precision, recall, and threshold tuning.
6. A model registry, deterministic canary split, tier routing, and cost arithmetic.
7. A typed plan validator and a safe fan-out pattern for independent specialist work.
8. A human approval route that pauses without holding a worker alive.
9. A financial crime alert triage pyramid where the model recommends, code disposes, and humans remain accountable.

### One-time setup

```bash
cd "chapter 5"
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
jupyter notebook
```

Jupyter supports top-level `await`, and this notebook uses it for the asynchronous examples.

## 0. Setup

The real chapter snippets use the OpenAI Agents SDK. The local notebook uses small teaching doubles so you can see the orchestration mechanics without needing a live endpoint. The names still match the chapter: agents produce typed outputs, runners have turn budgets, and the model registry lives in one place.

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

from pydantic import BaseModel, Field

from chapter5.models import MODELS


def line(title: str) -> None:
    print("\n" + title + "\n" + "-" * len(title))


def dump(model_or_value: Any) -> Any:
    if isinstance(model_or_value, BaseModel):
        return model_or_value.model_dump()
    return model_or_value


print(MODELS)


## 1. The Orchestration Dial

Chapter section: **The question every agent system answers by accident**.

The first control-flow decision is whether code owns the sequence or the model owns the sequence. In a code-led workflow, the model does a focused job and Python decides what happens next. That is the right shape when the branches are stable, testable, and consequential.

This invoice example mirrors the chapter snippet: the model-shaped extractor produces a typed invoice, but the review threshold is ordinary code.

In [ ]:
class ExtractedInvoice(BaseModel):
    vendor: str
    amount: float
    currency: str


class InvoiceOutcome(BaseModel):
    vendor: str
    amount: float
    currency: str
    route: Literal["ledger", "human_review"]
    reason: str


def extract_invoice_locally(raw_text: str) -> ExtractedInvoice:
    vendor_match = re.search(r"vendor[:=]\s*([A-Za-z0-9 &.-]+)", raw_text, re.I)
    amount_match = re.search(r"(USD|GBP|EUR)\s*([0-9,]+(?:\.[0-9]{2})?)", raw_text, re.I)
    if not vendor_match or not amount_match:
        raise ValueError("invoice text must include vendor and currency amount")
    return ExtractedInvoice(
        vendor=vendor_match.group(1).strip(),
        currency=amount_match.group(1).upper(),
        amount=float(amount_match.group(2).replace(",", "")),
    )


async def route_to_review(invoice: ExtractedInvoice) -> InvoiceOutcome:
    return InvoiceOutcome(
        vendor=invoice.vendor,
        amount=invoice.amount,
        currency=invoice.currency,
        route="human_review",
        reason="Amount exceeds the automatic posting threshold.",
    )


async def post_to_ledger(invoice: ExtractedInvoice) -> InvoiceOutcome:
    return InvoiceOutcome(
        vendor=invoice.vendor,
        amount=invoice.amount,
        currency=invoice.currency,
        route="ledger",
        reason="Amount is inside the automatic posting threshold.",
    )


async def process_invoice(raw_text: str) -> InvoiceOutcome:
    invoice = extract_invoice_locally(raw_text)
    if invoice.amount > 10_000:
        return await route_to_review(invoice)
    return await post_to_ledger(invoice)


for raw in [
    "vendor: Northwind Office Supply; total due USD 840.00",
    "vendor: Contoso Risk Analytics; total due USD 18750.00",
]:
    print(dump(await process_invoice(raw)))


Read the branch carefully. The model-shaped part extracts a structured invoice. The institution's policy, "over 10,000 goes to review", is not hidden in a prompt. It is code you can unit test.

## 2. Engineering The Loop You Already Own

Chapter section: **Engineering the loop you already own**.

Every run ends one of four ways: final output, handoff, exception, or a cap such as `max_turns`. The important habit is to make each ending part of the workflow contract, not an accidental stack trace.

In [ ]:
class MaxTurnsExceeded(Exception):
    pass


class ModelBehaviorError(Exception):
    pass


class InvestigationResult(BaseModel):
    case_id: str
    status: Literal["closed", "handed_off", "retry_queued", "escalated_to_human"]
    reason: str
    next_owner: str


@dataclass
class ScriptedInvestigator:
    name: str
    behavior: Literal["final", "handoff", "malformed", "wander"]

    async def run(self, case_id: str, summary: str, max_turns: int) -> InvestigationResult:
        if self.behavior == "malformed":
            raise ModelBehaviorError("structured output did not match InvestigationResult")
        for turn in range(1, max_turns + 1):
            if self.behavior == "final" and turn == 2:
                return InvestigationResult(
                    case_id=case_id,
                    status="closed",
                    reason="Evidence supports closing the case as expected activity.",
                    next_owner="workflow",
                )
            if self.behavior == "handoff" and turn == 1:
                return InvestigationResult(
                    case_id=case_id,
                    status="handed_off",
                    reason="Sanctions evidence needs specialist review.",
                    next_owner="sanctions_specialist",
                )
        raise MaxTurnsExceeded(f"{self.name} used {max_turns} turns without a conclusion")


async def run_investigation(
    case_id: str,
    summary: str,
    investigator: ScriptedInvestigator,
    max_turns: int = 3,
) -> InvestigationResult:
    try:
        return await investigator.run(case_id, summary, max_turns=max_turns)
    except MaxTurnsExceeded:
        return InvestigationResult(
            case_id=case_id,
            status="escalated_to_human",
            reason="Turn budget exhausted before the agent reached a conclusion.",
            next_owner="human_investigator",
        )
    except ModelBehaviorError:
        return InvestigationResult(
            case_id=case_id,
            status="retry_queued",
            reason="Model produced malformed output; queued for one retry.",
            next_owner="orchestrator_retry_queue",
        )


for behavior in ["final", "handoff", "malformed", "wander"]:
    result = await run_investigation(
        case_id=f"CASE-{behavior}",
        summary="Customer activity changed after a dormant period.",
        investigator=ScriptedInvestigator("investigator", behavior),
        max_turns=3,
    )
    print(behavior, dump(result))


A `MaxTurnsExceeded` branch should usually escalate rather than retry. An agent that spent its whole turn budget failing to conclude is telling you the task exceeded its design envelope. A malformed output can often get one controlled retry, but the retry policy belongs to the step, not to the whole pipeline.

## 3. Compound Reliability Decay

Chapter section: **Compound reliability decay, with actual numbers**.

Long chains fail even when each step looks good on its own. If each step has independent success probability `p`, a chain of `k` dependent steps succeeds at `p ** k`.

In [ ]:
def end_to_end_success(per_step_success: float, steps: int) -> float:
    return per_step_success**steps


rates = [0.99, 0.95, 0.90]
step_counts = [5, 10, 20]

header = "Per-step success" + "".join(f"{steps:>12} steps" for steps in step_counts)
print(header)
for rate in rates:
    row = f"{rate:>6.0%}".ljust(16)
    row += "".join(f"{end_to_end_success(rate, steps):>12.1%}" for steps in step_counts)
    print(row)


The engineering response is to shorten chains, validate boundaries, and measure real per-step rates. The next cell adds crash recovery to that same discipline.

## 4. Surviving A Crash At Step Seven

Chapter section: **Surviving a crash at step seven**.

A pipeline's state is more than a transcript. Checkpointing records which step completed, what output it produced, and which side effects already happened. On resume, the entry point starts from the last durable checkpoint instead of from zero.

In [ ]:
@dataclass
class CheckpointStore:
    checkpoints: dict[str, dict[str, Any]] = field(default_factory=dict)
    side_effects: set[str] = field(default_factory=set)

    def last_completed_index(self, case_id: str) -> int:
        return self.checkpoints.get(case_id, {}).get("completed_index", -1)

    def save(self, case_id: str, completed_index: int, output: dict[str, Any]) -> None:
        self.checkpoints[case_id] = {
            "completed_index": completed_index,
            "output": output,
        }

    def apply_side_effect_once(self, idempotency_key: str) -> str:
        if idempotency_key in self.side_effects:
            return "skipped_duplicate"
        self.side_effects.add(idempotency_key)
        return "applied"


async def run_alert_workflow(
    case_id: str,
    store: CheckpointStore,
    crash_after: str | None = None,
) -> dict[str, Any]:
    steps: list[tuple[str, Callable[[dict[str, Any]], dict[str, Any]]]] = [
        ("create_case_record", lambda state: {**state, "case_record": store.apply_side_effect_once(f"{case_id}:case_record")}),
        ("place_document_hold", lambda state: {**state, "document_hold": store.apply_side_effect_once(f"{case_id}:document_hold")}),
        ("screen_transactions", lambda state: {**state, "screening": "unusual_cash_pattern"}),
        ("summarize_case", lambda state: {**state, "summary": "Case ready for investigator."}),
    ]
    state = dict(store.checkpoints.get(case_id, {}).get("output", {}))
    start = store.last_completed_index(case_id) + 1
    for index, (name, step) in enumerate(steps[start:], start=start):
        state = step(state)
        store.save(case_id, index, state)
        if crash_after == name:
            raise RuntimeError(f"worker crashed after {name}")
    return state


checkpoint_store = CheckpointStore()
try:
    await run_alert_workflow("CASE-7", checkpoint_store, crash_after="screen_transactions")
except RuntimeError as err:
    print(err)

line("Checkpoint after crash")
print(checkpoint_store.checkpoints["CASE-7"])

line("Resume from checkpoint")
resumed = await run_alert_workflow("CASE-7", checkpoint_store)
print(resumed)
print("side effects:", sorted(checkpoint_store.side_effects))


The side effects were protected by idempotency keys, and the resumed run did not redo the completed work. Durable execution engines generalize this pattern, but the principle is the same: the orchestration layer should be boring, deterministic, and recoverable.

## 5. Routing Requests: Who Answers The Phone

Chapter section: **Routing requests: deciding who answers the phone**.

For high-volume tickets, emails, documents, and alerts, a classifier router is often stronger than a free-running agent. A small model-shaped classifier emits a typed route decision. Code applies the threshold and sends low-confidence or unknown work to a human.

In [ ]:
RouteCategory = Literal["balance_inquiry", "dispute", "fraud_report", "other"]


class RouteDecision(BaseModel):
    category: RouteCategory
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


class DispatchOutcome(BaseModel):
    category: RouteCategory | Literal["human_review"]
    output: str
    confidence: float


def classify_request(request: str) -> RouteDecision:
    text = request.lower()
    if any(word in text for word in ["stolen", "fraud", "unauthorized", "card cloned"]):
        return RouteDecision(category="fraud_report", confidence=0.91, reason="Fraud language appears in the request.")
    if any(word in text for word in ["dispute", "chargeback", "wrong charge", "merchant"]):
        return RouteDecision(category="dispute", confidence=0.86, reason="The customer challenges a posted transaction.")
    if any(word in text for word in ["balance", "available funds", "statement"]):
        return RouteDecision(category="balance_inquiry", confidence=0.88, reason="The customer asks about account information.")
    if any(word in text for word in ["wire", "beneficiary", "crypto", "urgent"]):
        return RouteDecision(category="fraud_report", confidence=0.62, reason="Risk terms appear, but the request is ambiguous.")
    return RouteDecision(category="other", confidence=0.40, reason="No known category matched cleanly.")


async def dispatch(request: str, threshold: float = 0.70) -> DispatchOutcome:
    decision = classify_request(request)
    if decision.category == "other" or decision.confidence < threshold:
        return DispatchOutcome(
            category="human_review",
            output=f"Queued for human review: {decision.reason}",
            confidence=decision.confidence,
        )
    responses = {
        "balance_inquiry": "Balance specialist will answer the account question.",
        "dispute": "Dispute specialist will own the transaction challenge.",
        "fraud_report": "Fraud specialist will secure the account and investigate.",
    }
    return DispatchOutcome(
        category=decision.category,
        output=responses[decision.category],
        confidence=decision.confidence,
    )


requests = [
    "What is my available balance after yesterday's deposit?",
    "I need to dispute a merchant charge from Northwind.",
    "My card was stolen and there are unauthorized purchases.",
    "Please change the wire beneficiary urgently.",
    "Can you explain your mobile app color theme?",
]

for request in requests:
    print(request)
    print(dump(await dispatch(request)))


The safety features are deliberate: one classifier decision, an `other` category, and a confidence threshold. A classifier without an "I do not know" route will confidently force novel inputs into familiar buckets.

## 6. Measuring The Router

Chapter section: **Trust, then verify: measuring the router**.

Routers are unusually easy to evaluate. Label a sample, run the router, then inspect where each true category went. The confusion matrix tells you which category definitions blur together and what your threshold trades.

In [ ]:
class LabeledRequest(BaseModel):
    text: str
    label: RouteCategory


evaluation_set = [
    LabeledRequest(text="What is my available balance today?", label="balance_inquiry"),
    LabeledRequest(text="Send me last month's statement.", label="balance_inquiry"),
    LabeledRequest(text="I want to dispute a restaurant charge.", label="dispute"),
    LabeledRequest(text="This merchant billed me twice.", label="dispute"),
    LabeledRequest(text="My card was cloned at an ATM.", label="fraud_report"),
    LabeledRequest(text="There are unauthorized transactions on my account.", label="fraud_report"),
    LabeledRequest(text="Please change my wire beneficiary urgently.", label="fraud_report"),
    LabeledRequest(text="How do I update my mailing address?", label="other"),
]


def evaluate_router(items: list[LabeledRequest], threshold: float) -> dict[str, Any]:
    matrix: dict[str, Counter[str]] = defaultdict(Counter)
    human_reviews = 0
    wrong_queue = 0
    for item in items:
        decision = classify_request(item.text)
        predicted = decision.category
        if decision.category == "other" or decision.confidence < threshold:
            predicted = "human_review"
            human_reviews += 1
        elif predicted != item.label:
            wrong_queue += 1
        matrix[item.label][predicted] += 1
    return {
        "threshold": threshold,
        "human_reviews": human_reviews,
        "wrong_queue": wrong_queue,
        "matrix": matrix,
    }


def print_matrix(report: dict[str, Any]) -> None:
    labels = ["balance_inquiry", "dispute", "fraud_report", "other", "human_review"]
    print(f"threshold={report['threshold']} human_reviews={report['human_reviews']} wrong_queue={report['wrong_queue']}")
    print("true / predicted".ljust(20) + "".join(label[:12].rjust(14) for label in labels))
    for true_label in labels[:-1]:
        row = true_label.ljust(20)
        row += "".join(str(report["matrix"][true_label][pred]).rjust(14) for pred in labels)
        print(row)


for threshold in [0.50, 0.70, 0.90]:
    print_matrix(evaluate_router(evaluation_set, threshold))
    print()


Raising the threshold sends more work to humans and less work to the wrong queue. Lowering it does the reverse. The right answer depends on the cost of a wrong route in your domain.

## 7. Routing Models

Chapter section: **Routing models: the cheapest big decision in your architecture**.

Model routing is an orchestration decision. The layer that knows the step also knows whether the step is extraction, conversation, planning, investigation, or formatting. Keep model strings in one registry and route by step identity, not by scattered inline names.

In [ ]:
class StepSpec(BaseModel):
    name: str
    task_family: Literal["classification", "extraction", "drafting", "planning", "investigation"]
    blast_radius: Literal["low", "medium", "high"]
    ambiguity: Literal["low", "medium", "high"]


class ModelRoute(BaseModel):
    model: str
    reasoning_effort: Literal["minimal", "medium", "high"]
    reason: str


def route_model(step: StepSpec) -> ModelRoute:
    if step.task_family in {"classification", "extraction"} and step.blast_radius == "low":
        return ModelRoute(model=MODELS.small, reasoning_effort="minimal", reason="Narrow, checkable task.")
    if step.task_family in {"drafting", "classification"} and step.ambiguity != "high":
        return ModelRoute(model=MODELS.mid, reasoning_effort="medium", reason="Routine language work with moderate judgment.")
    return ModelRoute(model=MODELS.frontier, reasoning_effort="high", reason="High ambiguity, planning, or consequential judgment.")


steps = [
    StepSpec(name="triage_alert", task_family="classification", blast_radius="low", ambiguity="low"),
    StepSpec(name="draft_customer_note", task_family="drafting", blast_radius="medium", ambiguity="medium"),
    StepSpec(name="investigate_sanctions_hit", task_family="investigation", blast_radius="high", ambiguity="high"),
]

for step in steps:
    print(step.name, dump(route_model(step)))


The goal is not to guess which model is cheapest. The goal is to start with quality, run evaluations, and then downshift one step at a time where quality holds.

### Pinning And Canarying

Chapter section: **Pinning model strings, and the day the default changed** and **Canarying a brain transplant**.

A model upgrade is a deploy. Use dated model strings, route a deterministic slice of traffic to the canary, and keep metrics split by model version.

In [ ]:
def pick_frontier_model(request_id: str, canary_share: float = 0.05) -> str:
    digest = hashlib.sha256(request_id.encode()).digest()
    bucket = int.from_bytes(digest[:4], "big") / 2**32
    return MODELS.frontier_canary if bucket < canary_share else MODELS.frontier


sample = [pick_frontier_model(f"request-{i}") for i in range(1_000)]
counts = Counter(sample)
print(counts)
print("canary share:", round(counts[MODELS.frontier_canary] / len(sample), 3))

for request_id in ["alert-1001", "alert-1001", "alert-1002"]:
    print(request_id, pick_frontier_model(request_id))


The same request id always lands on the same side. That makes comparisons stable and keeps user-visible behavior from flipping randomly between attempts.

### Cost Arithmetic

Chapter section: **The tier list** and **The cost arithmetic, and where it actually points**.

Prices move, so the notebook uses relative units. The lesson is the ratio: small-tier work on frontier models compounds into a budget line.

In [ ]:
RELATIVE_TOKEN_COST = {
    "small": 1.0,
    "mid": 8.0,
    "frontier": 25.0,
}


def relative_layer_cost(
    daily_items: int,
    input_tokens: int,
    output_tokens: int,
    tier: Literal["small", "mid", "frontier"],
) -> float:
    million_tokens = daily_items * (input_tokens + output_tokens) / 1_000_000
    return million_tokens * RELATIVE_TOKEN_COST[tier]


alerts_per_day = 10_000
triage_small = relative_layer_cost(alerts_per_day, 3_000, 300, "small")
triage_frontier = relative_layer_cost(alerts_per_day, 3_000, 300, "frontier")
investigation_frontier = relative_layer_cost(500, 20_000, 2_000, "frontier")

print({
    "triage_small_units_per_day": round(triage_small, 2),
    "triage_frontier_units_per_day": round(triage_frontier, 2),
    "investigation_frontier_units_per_day_at_5_percent_escalation": round(investigation_frontier, 2),
    "frontier_overpayment_ratio_for_triage": round(triage_frontier / triage_small, 1),
})


The investigation layer can dominate total cost even when far fewer items reach it. That means router accuracy is not just a quality metric. It directly controls the expensive layer's volume.

## 8. Plans, Supervisors, And The Coordination Tax

Chapter section: **The plan is data, not vibes**.

Planner-executor becomes useful when the plan is a typed artifact. Data can be validated before side effects occur, logged for auditors, and revised after a controlled failure.

In [ ]:
class PlanStep(BaseModel):
    description: str
    tool: str
    success_check: str


class Plan(BaseModel):
    goal: str
    steps: list[PlanStep]


def validate_plan(plan: Plan, allowed_tools: set[str], max_steps: int) -> list[str]:
    problems = [
        f"Step uses unknown or unauthorized tool: {step.tool}"
        for step in plan.steps
        if step.tool not in allowed_tools
    ]
    if len(plan.steps) > max_steps:
        problems.append(f"Plan has {len(plan.steps)} steps; budget is {max_steps}.")
    return problems


allowed_tools = {"lookup_transactions", "screen_sanctions", "summarize_case"}
valid_plan = Plan(
    goal="Prepare alert case file",
    steps=[
        PlanStep(description="Pull recent transactions", tool="lookup_transactions", success_check="Transactions loaded"),
        PlanStep(description="Check sanctions exposure", tool="screen_sanctions", success_check="Sanctions result attached"),
        PlanStep(description="Summarize evidence", tool="summarize_case", success_check="Case summary cites evidence"),
    ],
)
invalid_plan = Plan(
    goal="Prepare alert case file",
    steps=valid_plan.steps + [
        PlanStep(description="Email the customer directly", tool="send_customer_email", success_check="Email sent"),
        PlanStep(description="Close the alert", tool="close_alert", success_check="Alert closed"),
    ],
)

print("valid plan problems:", validate_plan(valid_plan, allowed_tools, max_steps=4))
print("invalid plan problems:", validate_plan(invalid_plan, allowed_tools, max_steps=4))


The plan check reaches into the orchestration layer before execution. Unknown tools and excessive step counts are rejected while the plan is still cheap to change.

### Fanning Out Without Falling Over

Chapter section: **Fanning out without falling over**.

Fan-out is for independent subtasks. If specialist B would do its job differently after reading specialist A's output, the work is not independent and should not be parallelized.

In [ ]:
class SpecialistView(BaseModel):
    specialist: str
    finding: str
    confidence: float


async def sanctions_screener(case_summary: str) -> SpecialistView:
    await asyncio.sleep(0.01)
    return SpecialistView(specialist="sanctions", finding="No direct sanctions match found.", confidence=0.82)


async def transaction_pattern_analyst(case_summary: str) -> SpecialistView:
    await asyncio.sleep(0.01)
    return SpecialistView(specialist="transactions", finding="Cash deposits are clustered below reporting thresholds.", confidence=0.77)


async def counterparty_network_mapper(case_summary: str) -> SpecialistView:
    await asyncio.sleep(0.01)
    return SpecialistView(specialist="network", finding="Counterparties share one newly opened address.", confidence=0.71)


async def gather_specialist_views(case_summary: str) -> list[SpecialistView]:
    runs = [
        sanctions_screener(case_summary),
        transaction_pattern_analyst(case_summary),
        counterparty_network_mapper(case_summary),
    ]
    return list(await asyncio.gather(*runs))


views = await gather_specialist_views("Dormant account with new cash activity and new counterparties.")
for view in views:
    print(dump(view))


Supervisor-worker systems add a larger coordination tax. Reach for them only when the task decomposes cleanly, the value justifies the token multiplier, and evaluations prove the multi-agent version beats the best single-agent version.

## 9. Humans Inside The Loop

Chapter section: **Putting humans inside the loop, not on top of it**.

A human approval is a route with a payload contract, an owner, and an SLA. Mechanically, the important requirement is pause without dying: serialize state, release the worker, and resume later when the decision arrives.

In [ ]:
class CaseFile(BaseModel):
    case_id: str
    summary: str
    proposed_action: Literal["close_alert", "file_sar"]
    risk_score: float


class ApprovalRequest(BaseModel):
    case_id: str
    action: str
    evidence: list[str]
    approve_effect: str
    reject_effect: str
    expires_at: str


class CaseStatus(BaseModel):
    case_id: str
    state: Literal["awaiting_approval", "closed", "escalated"]
    outcome: str | None = None


approval_queue: list[ApprovalRequest] = []
run_state: dict[str, dict[str, Any]] = {}


async def advance_case(case: CaseFile) -> CaseStatus:
    state = run_state.get(case.case_id, {"approval": None})
    if case.proposed_action == "file_sar" and state["approval"] is None:
        request = ApprovalRequest(
            case_id=case.case_id,
            action="file_sar",
            evidence=[case.summary, f"risk_score={case.risk_score}"],
            approve_effect="SAR package moves to analyst filing queue.",
            reject_effect="Case stays open for additional investigation.",
            expires_at="2026-06-28T17:00:00Z",
        )
        approval_queue.append(request)
        run_state[case.case_id] = {"approval": "pending", "request": request.model_dump()}
        return CaseStatus(case_id=case.case_id, state="awaiting_approval")
    if state["approval"] == "approved":
        return CaseStatus(case_id=case.case_id, state="closed", outcome="Approved SAR package queued for analyst filing.")
    if state["approval"] == "rejected":
        return CaseStatus(case_id=case.case_id, state="escalated", outcome="Rejected action returned for more investigation.")
    return CaseStatus(case_id=case.case_id, state="closed", outcome="Alert closed under policy.")


def record_human_decision(case_id: str, approved: bool) -> None:
    if case_id not in run_state or run_state[case_id].get("approval") != "pending":
        raise ValueError("no pending approval for case")
    run_state[case_id]["approval"] = "approved" if approved else "rejected"


case = CaseFile(
    case_id="AML-42",
    summary="Layered transfers through newly related counterparties.",
    proposed_action="file_sar",
    risk_score=0.91,
)
print(dump(await advance_case(case)))
print("approval package:", dump(approval_queue[-1]))
record_human_decision("AML-42", approved=True)
print(dump(await advance_case(case)))


The approval request leads with evidence and scoped effects. The human is not approving a vague permission. They are approving one action with named consequences.

## 10. Putting It Together: The Alert Triage Pyramid

Chapter section: **Putting it together: the alert triage pyramid**.

The financial crime example combines the whole chapter. Layer 1 triage is a small-model, single-turn classifier. Layer 2 investigation is a bounded frontier-model loop. Layer 3 is human analysis. The model recommends; policy code disposes; humans remain accountable for consequential decisions.

In [ ]:
AlertType = Literal["structuring", "sanctions", "known_false_positive", "new_typology"]


class Alert(BaseModel):
    alert_id: str
    alert_type: AlertType
    amount: float
    customer_tenure_years: int
    counterparty_country: str
    memo: str

    @property
    def case_text(self) -> str:
        return (
            f"{self.alert_type} alert for amount {self.amount}. "
            f"Customer tenure {self.customer_tenure_years} years. "
            f"Counterparty country {self.counterparty_country}. Memo: {self.memo}"
        )


class TriageVerdict(BaseModel):
    recommendation: Literal["clear", "investigate"]
    confidence: float = Field(ge=0.0, le=1.0)
    matched_typologies: list[str]
    evidence: list[str]
    rationale: str


class DispositionPolicy(BaseModel):
    auto_clear_threshold: float
    auto_clear_permitted: bool
    sample_rate: float = 0.05


class Disposition(BaseModel):
    alert_id: str
    route: Literal["auto_cleared", "human_confirm_clear", "investigation", "sampled_human_review"]
    model: str
    reason: str


DISPOSITION_POLICY: dict[AlertType, DispositionPolicy] = {
    "known_false_positive": DispositionPolicy(auto_clear_threshold=0.80, auto_clear_permitted=True, sample_rate=0.10),
    "structuring": DispositionPolicy(auto_clear_threshold=0.92, auto_clear_permitted=True, sample_rate=0.05),
    "sanctions": DispositionPolicy(auto_clear_threshold=1.00, auto_clear_permitted=False, sample_rate=0.00),
    "new_typology": DispositionPolicy(auto_clear_threshold=1.00, auto_clear_permitted=False, sample_rate=0.00),
}

sample_queue: list[tuple[Alert, TriageVerdict]] = []
investigation_queue: list[tuple[Alert, TriageVerdict]] = []
disposition_log: list[Disposition] = []


def triage_alert(alert: Alert) -> TriageVerdict:
    memo = alert.memo.lower()
    if alert.alert_type == "sanctions" or "sanction" in memo:
        return TriageVerdict(
            recommendation="investigate",
            confidence=0.94,
            matched_typologies=["sanctions_exposure"],
            evidence=["sanctions alert type", alert.counterparty_country],
            rationale="Sanctions alerts are never auto-cleared by policy.",
        )
    if alert.alert_type == "known_false_positive" and alert.customer_tenure_years >= 5:
        return TriageVerdict(
            recommendation="clear",
            confidence=0.88,
            matched_typologies=["known_false_positive_pattern"],
            evidence=["long-tenured customer", "historically explained pattern"],
            rationale="Pattern matches a previously reviewed false-positive family.",
        )
    if alert.alert_type == "structuring" and (alert.amount < 10_000 or "cash" in memo):
        return TriageVerdict(
            recommendation="investigate",
            confidence=0.79,
            matched_typologies=["possible_structuring"],
            evidence=["cash activity", f"amount={alert.amount}"],
            rationale="Cash activity near reporting thresholds needs investigation.",
        )
    return TriageVerdict(
        recommendation="clear",
        confidence=0.61,
        matched_typologies=[],
        evidence=["no known typology matched strongly"],
        rationale="Weak clear recommendation; policy should send this to confirmation.",
    )


def should_sample(alert_id: str, sample_rate: float) -> bool:
    digest = hashlib.sha256(alert_id.encode()).digest()
    bucket = int.from_bytes(digest[:4], "big") / 2**32
    return bucket < sample_rate


async def dispose_alert(alert: Alert) -> Disposition:
    verdict = triage_alert(alert)
    policy = DISPOSITION_POLICY[alert.alert_type]
    model = route_model(StepSpec(name="alert_triage", task_family="classification", blast_radius="low", ambiguity="low")).model

    if (
        verdict.recommendation == "clear"
        and verdict.confidence >= policy.auto_clear_threshold
        and policy.auto_clear_permitted
    ):
        if should_sample(alert.alert_id, policy.sample_rate):
            sample_queue.append((alert, verdict))
            disposition = Disposition(
                alert_id=alert.alert_id,
                route="sampled_human_review",
                model=model,
                reason="Auto-clear candidate sampled for permanent human review.",
            )
        else:
            disposition = Disposition(
                alert_id=alert.alert_id,
                route="auto_cleared",
                model=model,
                reason="Clear recommendation met policy threshold.",
            )
        disposition_log.append(disposition)
        return disposition

    if verdict.recommendation == "clear":
        sample_queue.append((alert, verdict))
        disposition = Disposition(
            alert_id=alert.alert_id,
            route="human_confirm_clear",
            model=model,
            reason="Clear recommendation did not meet the policy threshold.",
        )
        disposition_log.append(disposition)
        return disposition

    investigation_queue.append((alert, verdict))
    disposition = Disposition(
        alert_id=alert.alert_id,
        route="investigation",
        model=model,
        reason="Investigation recommendation or non-auto-clearable alert type.",
    )
    disposition_log.append(disposition)
    return disposition


alerts = [
    Alert(alert_id="ALERT-100", alert_type="known_false_positive", amount=120.00, customer_tenure_years=9, counterparty_country="GB", memo="recurring payroll memo"),
    Alert(alert_id="ALERT-101", alert_type="structuring", amount=9_850.00, customer_tenure_years=1, counterparty_country="GB", memo="cash deposit split"),
    Alert(alert_id="ALERT-102", alert_type="sanctions", amount=2_000.00, customer_tenure_years=3, counterparty_country="RU", memo="possible sanctions name match"),
    Alert(alert_id="ALERT-103", alert_type="new_typology", amount=700.00, customer_tenure_years=2, counterparty_country="GB", memo="novel pattern from upstream rule"),
]

for alert in alerts:
    print(dump(await dispose_alert(alert)))

print("sample_queue", len(sample_queue))
print("investigation_queue", len(investigation_queue))


The policy decisions live in code: whether an alert type may be auto-cleared, which threshold applies, and whether permanent sampling sends a clear candidate to human review. The model supplies a structured recommendation with evidence. It does not own disposition.

## 11. Operational Checks

Chapter section: **What breaks first**.

The pyramid needs ongoing checks for drift, novel alert types, injection-prone fields, and reviewer automation bias. The next helper turns the disposition log into a small operational snapshot.

In [ ]:
def operations_snapshot(dispositions: list[Disposition]) -> dict[str, Any]:
    by_route = Counter(item.route for item in dispositions)
    total = len(dispositions)
    investigation_rate = by_route["investigation"] / total if total else 0.0
    human_review_rate = (by_route["human_confirm_clear"] + by_route["sampled_human_review"]) / total if total else 0.0
    return {
        "total": total,
        "by_route": dict(by_route),
        "investigation_rate": round(investigation_rate, 3),
        "human_review_rate": round(human_review_rate, 3),
        "sample_queue_open": len(sample_queue) > 0,
    }


print(operations_snapshot(disposition_log))


A few percent of auto-cleared alerts should be sampled forever. That is not temporary QA. It is the guardrail against automation bias and quiet model drift.

## 12. Notebook Self-Check

Run this after executing the notebook from the top. It checks the important learning examples: invoice routing, loop endings, reliability math, checkpoint resume, request routing, model canarying, plan validation, fan-out, approval pause/resume, and alert disposition.

In [ ]:
# Invoice threshold
assert (await process_invoice("vendor: A; total due USD 9999.00")).route == "ledger"
assert (await process_invoice("vendor: B; total due USD 10001.00")).route == "human_review"

# Loop endings
assert (await run_investigation("C1", "summary", ScriptedInvestigator("i", "final"))).status == "closed"
assert (await run_investigation("C2", "summary", ScriptedInvestigator("i", "handoff"))).status == "handed_off"
assert (await run_investigation("C3", "summary", ScriptedInvestigator("i", "malformed"))).status == "retry_queued"
assert (await run_investigation("C4", "summary", ScriptedInvestigator("i", "wander"), max_turns=1)).status == "escalated_to_human"

# Reliability math
assert round(end_to_end_success(0.95, 10), 3) == 0.599

# Checkpoint resume should not duplicate side effects
fresh_store = CheckpointStore()
try:
    await run_alert_workflow("CASE-X", fresh_store, crash_after="place_document_hold")
except RuntimeError:
    pass
await run_alert_workflow("CASE-X", fresh_store)
assert fresh_store.side_effects == {"CASE-X:case_record", "CASE-X:document_hold"}

# Router behavior
assert (await dispatch("Please change my wire beneficiary urgently.")).category == "human_review"
assert (await dispatch("My card was stolen.")).category == "fraud_report"

# Canary split is deterministic
assert pick_frontier_model("same-request") == pick_frontier_model("same-request")

# Plan validation catches unknown tools and budget overflow
problems = validate_plan(invalid_plan, allowed_tools, max_steps=4)
assert any("unknown" in problem for problem in problems)
assert any("budget" in problem for problem in problems)

# Fan-out returns independent specialist views
assert len(await gather_specialist_views("case")) == 3

# Human route pauses and resumes
local_case = CaseFile(case_id="LOCAL-APPROVAL", summary="High risk", proposed_action="file_sar", risk_score=0.95)
assert (await advance_case(local_case)).state == "awaiting_approval"
record_human_decision("LOCAL-APPROVAL", approved=False)
assert (await advance_case(local_case)).state == "escalated"

# Triage policy keeps sanctions out of auto-clear
sanctions_alert = Alert(alert_id="CHECK-SANCTIONS", alert_type="sanctions", amount=100, customer_tenure_years=1, counterparty_country="RU", memo="sanctions")
assert (await dispose_alert(sanctions_alert)).route == "investigation"

print("Chapter 5 notebook self-check passed.")


## 13. Mapping Notebook Sections Back To The Chapter

- The orchestration dial: code owns stable consequences; model-led control is reserved for open-ended judgment.
- Engineering the loop: every run ends by final output, handoff, exception, or cap, and each ending has a typed outcome.
- Compound reliability: long chains multiply failure, so shorten chains and validate boundaries.
- Crash recovery: sessions help conversations, checkpoints help pipelines, and durable execution helps long side-effecting workflows.
- Request routing: a classifier router needs an `other` route, a threshold, and a measured confusion matrix.
- Model routing: model tier and thinking depth are orchestration choices, pinned and canaried like any dependency.
- Plans and supervisors: plans should be data, fan-out requires independence, and coordination must earn its tax.
- Human routes: approval is a durable route with scoped payloads, not a vague interruption.
- Alert triage pyramid: the model recommends, policy code disposes, and humans remain accountable.

The practical takeaway is the chapter's central control-plane claim: the question is not whether agents are powerful. The question is who owns each decision in the loop, and whether that ownership is visible, testable, recoverable, and measured.